# Optuna — búsqueda del λ_clDice óptimo (pesos Dice y CE fijos, como en v1)

**Nota metodológica.** Las búsquedas previas (7-8 trials) utilizaban una estructura de pérdida que replicaba, de forma no intencionada, el diseño de v2 (clDice calculado solo en resolución completa) en lugar del diseño original de v1 (clDice recalculado en cada escala de deep supervision, porque el módulo combinado Dice + CE + clDice se envuelve completo en `DeepSupervisionWrapper`). Esta versión implementa el trainer como réplica fiel de v1, variando únicamente λ.

**Objetivo único: maximizar `clDice`.** Dice, número de componentes y HD95 se registran en cada trial con carácter informativo.

`skeleton_iterations` se fija en 10 (valor por defecto de v1 y v2) para aislar el efecto de λ.

**Coste:** con un único hiperparámetro el espacio de búsqueda es mucho menor, por lo que 8 trials bastan para una estimación razonable del óptimo (frente a los 15 necesarios con 4 hiperparámetros). Con ≈ 69 s/época (mismo diseño de pérdida que v1, con clDice en todas las escalas de deep supervision) y 50 épocas por trial, cada trial requiere ≈ 76-78 min, es decir, **≈ 10-10.5 h de GPU en total para 8 trials**.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


In [2]:
!python -m pip install -q nnunetv2 optuna medpy scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

DATASET_ID = 501
DATASET_FOLDER_NAME = "Dataset501_ImageCAS"
CONFIGURATION = "3d_fullres"
FOLD = 0

# Valores fijos, como en v1: solo se busca lambda_cldice
WEIGHT_DICE_FIJO = 1.0
WEIGHT_CE_FIJO = 1.0
SKELETON_ITERATIONS_FIJO = 10

EPOCHS_POR_TRIAL = 50          # mismo criterio que notebook 22
TOTAL_TRIALS_DESEADOS = 16      # 15-20 recomendado para 1D con n_startup_trials=5
N_TRIALS_POR_SESION = 4          # ~78min/trial -> 4/sesion = ~5.2h, seguro en una sesion de Colab


GT_FOLDER = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations"

STUDY_NAME = "cldice_lambda_only_search_v3_fiel_v1"  # v2 del estudio (7 trials, topologia solo full-res) queda invalido: no replicaba v1
STORAGE_PATH = f"sqlite:///{BASE / 'optuna_lambda_only_study_v3.db'}"
CSV_RESULTADOS = BASE / "optuna_lambda_only_trials.csv"

print("Config lista.")

Config lista.


## 1. Funciones de métricas

Idénticas a las de los notebooks 19, 21 y 22.


In [4]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)
    return {
        "dice": dice_score(pred, gt),
        "cldice": hard_cldice(pred, gt),
        "cobertura_principal": cobertura_componente_principal(pred, gt)[0],
        "n_componentes": cobertura_componente_principal(pred, gt)[1],
        "hd95_crudo_mm": hd95_seguro(pred, gt, spacing),
    }


def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        filas.append(calcular_metricas_caso(pred_path, gt_path))

    df = pd.DataFrame(filas)
    return {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }

print("Funciones de metricas cargadas.")

Funciones de metricas cargadas.


## 2. Verificación con caso sintético


In [5]:
zz, yy, xx = np.mgrid[0:40, 0:15, 0:15]
tubo = (((yy - 7) ** 2 + (xx - 7) ** 2) <= 4 ** 2).astype(np.uint8)

print("dice:", dice_score(tubo, tubo))
print("cldice:", hard_cldice(tubo, tubo))
_, n = componentes_conexas(tubo)
print("n_componentes:", n)
print("OK: deberian salir dice~1.0, cldice~1.0, n_componentes=1")

dice: 0.999999999997449
cldice: 0.9999999947222222
n_componentes: 1
OK: deberian salir dice~1.0, cldice~1.0, n_componentes=1


## 3. Plantilla del trainer (solo varía `lambda_cldice`)

`weight_dice = 1.0`, `weight_ce = 1.0` y `skeleton_iterations = 10` permanecen fijos en cada trial, igual que en el diseño original de v1, de modo que λ es el único grado de libertad.


In [6]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    """Replica EXACTA de v1 (notebook 12): combina Dice+CE+clDice en un unico
    modulo por escala, y ese modulo COMPLETO se envuelve en DeepSupervisionWrapper
    -- por eso opera sobre tensores individuales (logits, target), no sobre listas.
    DeepSupervisionWrapper lo invoca una vez por cada escala de deep supervision,
    de modo que clDice se recalcula de forma independiente en cada una."""

    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.soft_cldice(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[Optuna lambda-only trial - fiel a v1] lambda_cldice={{self.lambda_cldice:.4f}} "
            f"(weight_dice={{self.weight_dice}}, weight_ce={{self.weight_ce}}, "
            f"skeleton_iterations={{self.skeleton_iterations}} -- fijos)"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

                # 1. Dice + CE + clDice combinados en un único módulo (como en v1)
        loss = DiceCEPlusSoftClDice(
            regional_loss=regional_loss,
            lambda_cldice=self.lambda_cldice,
            skeleton_iterations=self.skeleton_iterations,
        )

                # 2. Módulo combinado completo envuelto en DeepSupervisionWrapper (como en v1);
                #    clDice se recalcula en cada escala, no solo en resolución completa
        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)

        return loss
'''


def escribir_trainer(class_name, epochs, lambda_cldice):
    codigo = TRAINER_TEMPLATE.format(
        class_name=class_name,
        epochs=epochs,
        weight_dice=WEIGHT_DICE_FIJO,
        weight_ce=WEIGHT_CE_FIJO,
        lambda_cldice=lambda_cldice,
        skeleton_iterations=SKELETON_ITERATIONS_FIJO,
    )
    ruta = CUSTOM_TRAINERS / f"{class_name}.py"
    ruta.write_text(codigo)
    return ruta

print("Plantilla de trainer lista (solo lambda_cldice varia).")

Plantilla de trainer lista (solo lambda_cldice varia).


## 4. Función objetivo de Optuna

**Objetivo único: maximizar `clDice`.** Dice, n_componentes y HD95 se calculan y guardan en el CSV como métricas informativas, para analizar el trial ganador y desempatar entre valores de λ con clDice similar, sin condicionar la búsqueda.

Rango de búsqueda: `lambda_cldice ∈ [0.05, 0.5]`, con `weight_dice = 1.0` y `weight_ce = 1.0` fijos (como en v1).


In [7]:
import subprocess
import optuna


def objective(trial: optuna.Trial):
    lambda_cldice = trial.suggest_float("lambda_cldice", 0.05, 0.5)

    class_name = f"nnUNetTrainerOptunaLambdaV1Fiel_trial{trial.number}_{EPOCHS_POR_TRIAL}epochs"

    escribir_trainer(class_name=class_name, epochs=EPOCHS_POR_TRIAL, lambda_cldice=lambda_cldice)

    comando = [
        "nnUNetv2_train", str(DATASET_ID), CONFIGURATION, str(FOLD),
        "-tr", class_name, "--npz",
    ]

    print(f"\n=== Trial {trial.number}: lambda_cldice={lambda_cldice:.4f} "
          f"(weight_dice=1.0, weight_ce=1.0, iters=10 fijos) ===")

    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        print(resultado.stdout[-2000:])
        print(resultado.stderr[-2000:])
        raise RuntimeError(f"Entrenamiento del trial {trial.number} fallo")

    metricas = evaluar_trainer(class_name)

    fila = {
        "trial": trial.number,
        "lambda_cldice": lambda_cldice,
        **metricas,
    }
    pd.DataFrame([fila]).to_csv(
        CSV_RESULTADOS, mode="a",
        header=not CSV_RESULTADOS.exists(), index=False,
    )

    print(f"    -> Dice={metricas['dice_mean']:.4f}  clDice={metricas['cldice_mean']:.4f}  "
          f"n_comp={metricas['n_componentes_mean']:.2f}  HD95={metricas['hd95_crudo_mean']:.2f}mm")

    return metricas["cldice_mean"]

print("Funcion objetivo lista (objetivo unico: clDice, sin restriccion).")

Funcion objetivo lista (objetivo unico: clDice, sin restriccion).


## 5. Previsualización del espacio de búsqueda (sin entrenamiento)


In [8]:
estudio_preview = optuna.create_study(direction="maximize")

filas_preview = []
for _ in range(20):
    trial = estudio_preview.ask()
    lambda_cldice = trial.suggest_float("lambda_cldice", 0.05, 0.5)
    filas_preview.append({"lambda_cldice": round(lambda_cldice, 3)})

df_preview = pd.DataFrame(filas_preview).sort_values("lambda_cldice").reset_index(drop=True)
print(df_preview.to_string(index=False))
print()
print("Rango:", df_preview["lambda_cldice"].min(), "-", df_preview["lambda_cldice"].max())

[I 2026-09-15 13:38:47,177] A new study created in memory with name: no-name-9e2fb426-f9e9-4ecc-be49-a434b55eca7e


 lambda_cldice
         0.098
         0.157
         0.189
         0.210
         0.279
         0.283
         0.289
         0.312
         0.313
         0.315
         0.319
         0.350
         0.375
         0.386
         0.400
         0.412
         0.415
         0.446
         0.450
         0.478

Rango: 0.098 - 0.478


## 6. Ejecución del estudio (sampler explícito, reanudable entre trials)

Se define de forma explícita `sampler=TPESampler(n_startup_trials=5, seed=42)` en lugar del sampler por defecto de Optuna. Con `n_startup_trials = 5` (frente a los 10 por defecto) y 16 trials totales, 11 trials se benefician de la fase guiada por TPE y no solo del muestreo inicial.


In [9]:
from optuna.samplers import TPESampler

sampler = TPESampler(n_startup_trials=5, seed=42)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_PATH,
    load_if_exists=True,
    direction="maximize",
    sampler=sampler,
)

completados = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Trials completados hasta ahora: {completados} / {TOTAL_TRIALS_DESEADOS}")

trials_restantes = max(0, TOTAL_TRIALS_DESEADOS - completados)
trials_esta_sesion = min(N_TRIALS_POR_SESION, trials_restantes)
print(f"Trials a lanzar en esta sesion: {trials_esta_sesion}")

if trials_restantes == 0:
    print("Se alcanzó el total de trials previsto. El mejor trial se revisa en la celda siguiente.")

[I 2026-09-15 13:38:49,075] A new study created in RDB with name: cldice_lambda_only_search_v3_fiel_v1


Trials completados hasta ahora: 0 / 16
Trials a lanzar en esta sesion: 4


In [10]:
if trials_esta_sesion > 0:
    study.optimize(objective, n_trials=trials_esta_sesion)

df_todos = pd.read_csv(CSV_RESULTADOS) if CSV_RESULTADOS.exists() else pd.DataFrame()

print("\n=== Mejor trial hasta ahora (mayor clDice) ===")
print(f"Trial {study.best_trial.number}: lambda_cldice={study.best_trial.params['lambda_cldice']:.4f} "
      f"-> clDice={study.best_trial.value:.4f}")

if not df_todos.empty:
    fila_mejor = df_todos[df_todos['trial'] == study.best_trial.number]
    if not fila_mejor.empty:
        r = fila_mejor.iloc[0]
        print(f"    Dice={r['dice_mean']:.4f}  n_comp={r['n_componentes_mean']:.2f}  "
              f"HD95={r['hd95_crudo_mean']:.2f}mm")

print("\n=== Todos los trials completados hasta ahora (ordenados por clDice) ===")
if not df_todos.empty:
    print(df_todos.sort_values('cldice_mean', ascending=False).to_string(index=False))

print(f"\nResultados completos: {CSV_RESULTADOS}")
print(f"Estudio (reanudable): {STORAGE_PATH}")


=== Trial 0: lambda_cldice=0.2185 (weight_dice=1.0, weight_ce=1.0, iters=10 fijos) ===


[I 2026-09-15 15:47:10,817] Trial 0 finished with value: 0.7776678219421912 and parameters: {'lambda_cldice': 0.21854305348131314}. Best is trial 0 with value: 0.7776678219421912.


    -> Dice=0.7159  clDice=0.7777  n_comp=23.05  HD95=28.26mm

=== Trial 1: lambda_cldice=0.4778 (weight_dice=1.0, weight_ce=1.0, iters=10 fijos) ===


[I 2026-09-15 17:31:53,206] Trial 1 finished with value: 0.7727126759434396 and parameters: {'lambda_cldice': 0.4778214378844623}. Best is trial 0 with value: 0.7776678219421912.


    -> Dice=0.7123  clDice=0.7727  n_comp=24.62  HD95=29.98mm

=== Trial 2: lambda_cldice=0.3794 (weight_dice=1.0, weight_ce=1.0, iters=10 fijos) ===


[I 2026-09-15 19:16:48,646] Trial 2 finished with value: 0.7773383807072681 and parameters: {'lambda_cldice': 0.3793972738151323}. Best is trial 0 with value: 0.7776678219421912.


    -> Dice=0.7105  clDice=0.7773  n_comp=24.35  HD95=30.72mm

=== Trial 3: lambda_cldice=0.3194 (weight_dice=1.0, weight_ce=1.0, iters=10 fijos) ===


[I 2026-09-15 21:01:55,418] Trial 3 finished with value: 0.7889279497098889 and parameters: {'lambda_cldice': 0.31939631788866646}. Best is trial 3 with value: 0.7889279497098889.


    -> Dice=0.7185  clDice=0.7889  n_comp=21.48  HD95=28.45mm


ParserError: Error tokenizing data. C error: Expected 7 fields in line 10, saw 8


In [11]:
# 1. Inspección del archivo línea por línea
with open(CSV_RESULTADOS) as f:
    lineas = f.readlines()

for i, linea in enumerate(lineas):
    n_campos = len(linea.strip().split(","))
    print(i, n_campos, linea.strip()[:80])

0 7 trial,lambda_cldice,dice_mean,cldice_mean,n_componentes_mean,hd95_crudo_mean,n_c
1 7 0,0.3873329177517732,0.7135368412916236,0.7824974596328826,24.725,29.30771956267
2 7 1,0.3463597697248767,0.7028051076982994,0.7721372064128291,27.85,32.352714214834
3 7 2,0.4393663735275541,0.706026235053408,0.777082675145344,25.75,31.38799511887757
4 7 3,0.38085199619498433,0.7118738159319901,0.7775166357335298,25.225,30.4729154817
5 7 4,0.3994595291508219,0.7162152643300216,0.7846022011633187,24.0,28.0298158938088
6 7 5,0.47557484217586227,0.7184496974255081,0.7928916640477232,22.8,27.424654971197
7 7 6,0.12904876098854445,0.725458909313119,0.7927994905187921,22.55,26.860008814335
8 7 7,0.22730487995952836,0.7066286973707558,0.7694166763163098,25.75,31.01691903538
9 8 1,0.21854305348131314,False,0.7153685460187109,0.778859256124591,24.975,30.57751
10 8 2,0.4778214378844623,False,0.7038038494248801,0.7756651186940441,23.875,32.32930
11 8 3,0.3793972738151323,False,0.7195477273773381,0.78780457783

In [12]:
import pandas as pd

# 2. Selección de las filas del esquema nuevo (7 columnas: trial, lambda_cldice,
#    dice_mean, cldice_mean, n_componentes_mean, hd95_crudo_mean, n_casos)
columnas_correctas = ["trial", "lambda_cldice", "dice_mean", "cldice_mean",
                       "n_componentes_mean", "hd95_crudo_mean", "n_casos"]

filas_buenas = []
for linea in lineas:
    campos = linea.strip().split(",")
    if len(campos) == 7 and campos[0] != "trial":  # descarta cabeceras repetidas
        filas_buenas.append(campos)

df_limpio = pd.DataFrame(filas_buenas, columns=columnas_correctas)
df_limpio = df_limpio.apply(pd.to_numeric, errors="ignore")
print(df_limpio)

# 3. Sobrescritura del CSV ya depurado, con una única cabecera
df_limpio.to_csv(CSV_RESULTADOS, index=False)
print("CSV reescrito y limpio.")

    trial  lambda_cldice  dice_mean  cldice_mean  n_componentes_mean  \
0       0       0.387333   0.713537     0.782497              24.725   
1       1       0.346360   0.702805     0.772137              27.850   
2       2       0.439366   0.706026     0.777083              25.750   
3       3       0.380852   0.711874     0.777517              25.225   
4       4       0.399460   0.716215     0.784602              24.000   
5       5       0.475575   0.718450     0.792892              22.800   
6       6       0.129049   0.725459     0.792799              22.550   
7       7       0.227305   0.706629     0.769417              25.750   
8       5       0.218543   0.714741     0.780182              23.250   
9       6       0.066176   0.721257     0.787120              22.225   
10      0       0.218543   0.715887     0.777668              23.050   
11      1       0.477821   0.712330     0.772713              24.625   
12      2       0.379397   0.710487     0.777338              24

/tmp/ipykernel_965/1168207124.py:15: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_limpio = df_limpio.apply(pd.to_numeric, errors="ignore")


In [13]:
# Las últimas 4 filas corresponden a la arquitectura correcta (v1 fiel), confirmado
# porque coinciden con el log impreso de esta sesión
df_limpio_v1fiel = df_limpio.tail(4).reset_index(drop=True)
print(df_limpio_v1fiel)

df_limpio_v1fiel.to_csv(CSV_RESULTADOS, index=False)
print("CSV reescrito con SOLO los 4 trials de la arquitectura v1-fiel.")

   trial  lambda_cldice  dice_mean  cldice_mean  n_componentes_mean  \
0      0       0.218543   0.715887     0.777668              23.050   
1      1       0.477821   0.712330     0.772713              24.625   
2      2       0.379397   0.710487     0.777338              24.350   
3      3       0.319396   0.718497     0.788928              21.475   

   hd95_crudo_mean  n_casos  
0        28.257162       40  
1        29.979831       40  
2        30.721493       40  
3        28.448857       40  
CSV reescrito con SOLO los 4 trials de la arquitectura v1-fiel.


In [14]:
df_limpio_v1fiel.to_csv(BASE / "optuna_lambda_only_trials_cldice_lambda_only_search_v3_fiel_v1.csv", index=False)

In [15]:
import shutil
shutil.copy(CSV_RESULTADOS, BASE / "optuna_lambda_only_trials_cldice_lambda_only_search_v3_fiel_v1.csv")

PosixPath('/content/drive/MyDrive/TFM_nnUNet/optuna_lambda_only_trials_cldice_lambda_only_search_v3_fiel_v1.csv')

## 7. Limpieza opcional de carpetas huérfanas de trials interrumpidos


In [16]:
import re

carpetas_en_disco = sorted(NNUNET_RESULTS.glob(f"{DATASET_FOLDER_NAME}/nnUNetTrainerOptunaLambdaV1Fiel_trial*"))
trials_completados_ok = {
    t.number for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
}

huerfanas = []
for carpeta in carpetas_en_disco:
    m = re.search(r"trial(\d+)_", carpeta.name)
    if m and int(m.group(1)) not in trials_completados_ok:
        huerfanas.append(carpeta)

if not huerfanas:
    print("No hay carpetas huerfanas.")
else:
    print("Carpetas huerfanas encontradas:")
    for c in huerfanas:
        print(" -", c)
    print("\nDescomenta para borrar y liberar espacio:")
    # import shutil
    # for c in huerfanas:
    #     shutil.rmtree(c, ignore_errors=True)

No hay carpetas huerfanas.


## 8. Siguiente paso

Una vez identificado el λ ganador (único no dominado, o el que ofrezca mejor equilibrio dentro del frente de Pareto), se reentrena con 100 épocas completas, igual que en el notebook 22 con la configuración de cuatro hiperparámetros, para compararlo en igualdad de condiciones con baseline, v1, v2 y Optuna-Best.
